# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. The FAIR^2 dataset includes clinicopathological and molecular data for 77 cancer survivors with second primary colorectal cancer.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All referencing uses `@id` fields for all entities. Let's list the available RecordSets and their Fields.

In [ ]:
# List all available RecordSets and their schema

record_sets = dataset.record_sets

print(f"Found {len(record_sets)} record set(s) in the dataset.\n")
for rs in record_sets:
    print(f"RecordSet name: {rs.name} (ID: {rs.id})")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (ID: {field.id}) [Type: {field.data_type}]" )
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

**Note:** All `@id`s are referenced explicitly in code for reproducibility.

In [ ]:
# Extract data from all available RecordSets by their `@id`
import pprint

all_record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for rs_id in all_record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from RecordSet: {rs_id}")

# Display the first RecordSet (primary data) details
if all_record_set_ids:
    first_rs_id = all_record_set_ids[0]
    print(f"\nColumns in RecordSet '{first_rs_id}':")
    pprint.pprint(dataframes[first_rs_id].columns.tolist())
    print("\nPreview:")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records, normalizing numeric fields, and grouping by attributes. We use `@id` fields for all references.

### Example: Analyze Age Distribution by MSI Status

- We choose numeric field `cr:field:age` (replace with the actual `@id` from your overview).
- We group by categorical field `cr:field:msi_status`.

*Note*: The actual field IDs are discovered in the previous overview code.

In [ ]:
#--- Customize the field IDs below to match your dataset overview ---#
record_set_id = first_rs_id  # Use the primary data record set from above
# Set field IDs based on the printed overview. Example (change as appropriate):
numeric_field_id = 'cr:field:age'            # example: age
group_field_id = 'cr:field:msi_status'       # example: microsatellite instability status

df = dataframes[record_set_id]

# Make sure the chosen fields exist
assert numeric_field_id in df.columns, f"Field {numeric_field_id} not found in columns: {df.columns.tolist()}"
assert group_field_id in df.columns, f"Field {group_field_id} not found in columns: {df.columns.tolist()}"

# Handle missing or invalid entries
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
valid = df[numeric_field_id].notna()
df_valid = df[valid]

# Remove outliers (e.g., Age > 100 or < 10, domain-specific decision)
filtered_df = df_valid[df_valid[numeric_field_id].between(10, 100)]
print(f"Filtered records by age between 10 and 100. Remaining: {len(filtered_df)}.")

# Normalize the age field
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)

print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized", group_field_id]].head())

# Group by MSI status (example, using group_field_id)
if group_field_id in filtered_df.columns:
    grp = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'median', 'count'])
    print(f"\nGrouped statistics by {group_field_id}:")
    print(grp)
else:
    print(f"Group field {group_field_id} not found in DataFrame.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We plot Age distribution colored by MSI Status (`cr:field:msi_status`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of Age segmented by MSI status
plt.figure(figsize=(8, 5))
sns.histplot(
    data=filtered_df,
    x=numeric_field_id,
    hue=group_field_id,
    bins=15,
    kde=True,
    element="step",
    palette="Set2"
)
plt.xlabel('Age')
plt.ylabel('Count')
plt.title('Age Distribution by MSI Status')
plt.tight_layout()
plt.show()

# Boxplot by MSI status
plt.figure(figsize=(7, 5))
sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df, palette="Set2")
plt.xlabel('MSI Status')
plt.ylabel('Age')
plt.title('Age by MSI Status')
plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset provides comprehensive clinicopathological and molecular data for secondary primary colorectal cancer survivors.
- We loaded the Croissant schema, explored available record sets, and extracted the main data table using `@id` references throughout.
- Exploratory analysis and visualizations show the age distribution segmented by MSI status, enabling data-driven biomarker and risk stratification insights.

Further analysis can extend to other fields and groupings, leveraging the full Croissant schema with transparent use of `@id` fields.